# La reconstrucción como proyección ortogonal

Continuaremos utilizando $A_7$ y las mediciones con $\sigma = 0.06$ del ejercicio anterior. La reconstrucción por mínimos cuadrados $\hat{x}_\sigma$ no reproduce exactamente las mediciones ruidosas, sino que genera el vector $A_7\hat{x}_\sigma$ más cercano a $b_\sigma$ dentro de $\text{Col}(A_7)$.

In [ ]:

import numpy as np
import matplotlib.pyplot as plt


# Fantoma del ejercicio 1 (matriz y vector)
X_fantoma = np.array([
[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0],
[0.0, 0.0, 0.8, 0.3, 0.3, 0.3, 0.0, 0.0],
[0.0, 0.8, 0.8, 0.8, 0.3, 0.3, 0.3, 0.0],
[0.0, 0.3, 0.8, 0.3, 0.3, 0.3, 0.3, 0.0],
[0.0, 0.3, 0.3, 0.3, 0.3, 0.5, 0.3, 0.0],
[0.0, 0.3, 0.3, 0.3, 0.5, 0.5, 0.5, 0.0],
[0.0, 0.0, 0.3, 0.3, 0.3, 0.5, 0.0, 0.0],
[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0],
], dtype=float)
x_v_fantoma = X_fantoma.reshape(-1)


# Funciones de los ejercicios 2 y 3 (copiadas para que el notebook sea autocontenido)
def mascara_rayo(n, direccion, s):
    M = np.zeros((n, n))
    i, j = np.ogrid[:n, :n]
    condicion = (i * direccion[0] + j * direccion[1] == s)   # píxeles cuyo centro cumple α·i + β·j = s
    M[condicion] = 1
    return M


def matriz_proyeccion(n, direcciones):
    i, j = np.indices((n, n))
    filas = []    # filas de A, una por rayo
    rayos = []    # par ((α, β), s) de cada fila
    for (alfa, beta) in direcciones:
        S = alfa * i + beta * j               # a qué rayo pertenece cada píxel
        for s in np.unique(S):                # cada valor distinto de s es un rayo
            M = mascara_rayo(n, (alfa, beta), s)
            filas.append(M.reshape(-1))       # máscara vectorizada -> fila de A
            rayos.append(((alfa, beta), int(s)))
    A = np.array(filas, dtype=float)
    return A, rayos


# Matriz A7 del ejercicio 4
D7 = [(1, 0), (0, 1), (1, 1), (1, -1), (2, 1), (1, 2), (3, 1)]
A7, rayos7 = matriz_proyeccion(8, D7)

# Mediciones ideales (sin ruido)
b7 = A7 @ x_v_fantoma


# Mediciones con ruido sigma = 0.06 del ejercicio 6 (misma semilla y mismo z)
rng = np.random.default_rng(42)
z = rng.standard_normal(b7.shape[0])
escala = np.sqrt(np.mean(b7**2))
sigma = 0.06
b_sigma = b7 + sigma * escala * z

#### a) Calcular el residuo $r_\sigma = b_\sigma - A_7\hat{x}_\sigma$ y verificar numéricamente que $\|A_7^T r_\sigma\| \approx 0$.

In [5]:
# Reconstrucción por mínimos cuadrados
x_hat, _, _, _ = np.linalg.lstsq(A7, b_sigma, rcond=None)

# Residuo
r_sigma = b_sigma - A7 @ x_hat

print(f"||r_sigma||      = {np.linalg.norm(r_sigma):.4f}")
print(f"||A7ᵀ r_sigma||  = {np.linalg.norm(A7.T @ r_sigma):.2e}")

||r_sigma||      = 0.3511
||A7ᵀ r_sigma||  = 3.95e-14


El residuo no es cero ($\|r_\sigma\| \approx 0.35$), porque con ruido el sistema no tiene solución exacta, como se vio en el ejercicio 6. Sin embargo,  $\|A_7^T r_\sigma\| $ =  3.95e-14 (prácticamente cero).

#### b) Explicar por qué el resultado anterior indica que el residuo es ortogonal a todas las columnas de $A_7$ y que $A_7\hat{x}_\sigma$ es la proyección ortogonal de $b_\sigma$ sobre $\text{Col}(A_7)$.

Las filas de $A_7^T$ son las columnas $a_1, \dots, a_{64}$ de $A_7$, así que cada componente del vector $A_7^T r_\sigma$ es el producto punto entre una columna y el residuo

$$A_7^T r_\sigma = \begin{pmatrix} a_1^T r_\sigma \\ \vdots \\ a_{64}^T r_\sigma \end{pmatrix} \approx \begin{pmatrix} 0 \\ \vdots \\ 0 \end{pmatrix}$$

Como todos los productos punto son cero, $r_\sigma$ es ortogonal a cada columna de $A_7$(por definicion de ortogonalidad). Por ende, también es ortogonal a cualquier combinación lineal de ellas, es decir, a todo el subespacio $\text{Col}(A_7)$, el residuo pertenece al complemento ortogonal $\text{Col}(A_7)^\perp$, que es $\text{Nul}(A_7^T)$.

Las mediciones con ruido se pueden escribir como

$$b_\sigma = {A_7\hat{x}_\sigma} + {r_\sigma} $$

El primer término está en $\text{Col}(A_7)$, porque es una combinación lineal de las columnas de $A_7$, y el segundo es ortogonal a ese subespacio. Por el teorema de la descomposición ortogonal, esta forma de escribir $b_\sigma$ es única, y el término que está en el subespacio es justamente la proyección ortogonal de $b_\sigma$ sobre $\text{Col}(A_7)$. Además, como vimos en la Clase, la proyección ortogonal es el punto del subespacio más cercano a $b_\sigma$, que es lo que busca mínimos cuadrados.




#### c) Explicar la diferencia entre las proyecciones tomográficas medidas por el escáner y la proyección ortogonal que aparece en mínimos cuadrados.

Las proyecciones tomográficas son el conjunto de mediciones que obtiene el escáner para un ángulo fijo, al recorrer las distintas posiciones $s$ del detector. Para cada posición, el rayo atraviesa la imagen y el detector registra la atenuación acumulada a lo largo de su recorrido. En el modelo binario, el rayo selecciona los píxeles cuyo centro cumple $\alpha i + \beta j = s$, y la medición es la suma de esos píxeles.

Es decir, la ecuación $\alpha i + \beta j = s$ dice qué píxeles atraviesa cada rayo, y la proyección tomografica es la lista de todas estas sumas para una misma dirección. La proyección tomográfica lleva la imagen ($\mathbb{R}^{64}$) a las mediciones ($\mathbb{R}^{119}$). En conclusion, es lo que mide el escáner.

En cambio la proyección ortogonal es una de los conceptos vistos en la Clase.  Que lleva un vector al punto más cercano de un subespacio. Para eso, descompone el vector en una parte dentro del subespacio y otra ortogonal a él. En mínimos cuadrados, el vector es $b_\sigma$ y el subespacio es $\text{Col}(A_7)$ La parte que está en el subespacio, $A_7\hat{x}_\sigma$, es la proyección ortogonal de $b_\sigma$, y se puede calcular con la matriz de proyección 

$$A_7\hat{x}_\sigma = A_7\,(A_7^TA_7)^{-1}A_7^T\, b_\sigma$$

La parte que queda afuera es el residuo, que es ortogonal a todas las columnas de $A_7$. 

A diferencia de la proyección tomográfica, la proyección ortogonal no va de la imagen a las mediciones, sino que trabaja dentro del espacio de las mediciones. Es decir toma las mediciones con ruido $b_\sigma$ y las lleva a $A_7\hat{x}_\sigma$, que son las mediciones de $\text{Col}(A_7)$ más cercanas, es decir, las más parecidas a $b_\sigma$ que sí pueden ser producidas por alguna imagen. Aparece en la etapa de reconstrucción, y es lo que calcula mínimos cuadrados.

